In [109]:
from classifier_preprocess import prepare_fashion_data
import pyarrow.parquet as pq
import pandas as pd

# fashion_pq = pd.read_parquet('fashion_mentions.parquet')
# character_pq = pd.read_parquet('characters.parquet')
fashion_pq = pq.ParquetFile("fashion_mentions.parquet")
# characters_pq = pq.ParquetFile("characters.parquet")


characters_pq = pq.ParquetFile("characters.parquet")
all_character_chunks = []


all_character_chunks = []

for i in range(characters_pq.num_row_groups):
    df = characters_pq.read_row_group(i).to_pandas()
    # Only keep relevant columns
    df = df[['character_id', 'gender']]
    all_character_chunks.append(df)

# Combine and deduplicate by character_id (keeping first)
character_df = pd.concat(all_character_chunks, ignore_index=True)
character_df = character_df.drop_duplicates(subset='character_id')

# Save slimmed, deduplicated data
character_df.to_parquet("slim_characters.parquet", index=False)

slim_characters = pq.ParquetFile("slim_characters.parquet")

for i in range(fashion_pq.num_row_groups):
    fashion_df = fashion_pq.read_row_group(i).to_pandas()
    fashion_df['character_id'] = fashion_df['character_id'].astype(str)

    # Filter fashion_df before merge
    # fashion_df = fashion_df[fashion_df['gender'] != 'they/them/theirs']

    # Now merge with slim character data in a streaming-safe way
    # Step 1: Load slim_characters into a DataFrame (should be small enough now)
    slim_df = pd.concat([
        slim_characters.read_row_group(j).to_pandas()
        for j in range(slim_characters.num_row_groups)
    ])
    slim_df['character_id'] = slim_df['character_id'].astype(str)

    merged_df = fashion_df.merge(slim_df, on='character_id', how='left')





In [110]:
merged_df

,book_id,mention_id,term,sentence,start_idx,end_idx,sentence_start_idx,sentence_end_idx,adjectives,character_id,gender
0,wu.89097998645.clean,6971477,sheath,I set her down \nas a New England woman the fi...,119,125,173434,173598,[],2947,he/him/his
1,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],282,she/her
2,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],281,he/him/his
3,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],2947,he/him/his
4,wu.89097998645.clean,6971480,coat,"While I was mourning over my cloak, she \ncame...",163,167,174093,174336,[],186,she/her
...,...,...,...,...,...,...,...,...,...,...,...
329372,yale.39002088679528.clean,15208824,clothing,But instantly it seemed \nto me that the voice...,122,130,896937,897183,[bare],None,NaN
329373,yale.39002088679528.clean,15208826,garments,"And they answered that the other women also, \...",152,160,899605,899838,[],None,NaN
329374,yale.39002088679528.clean,15208827,linen,And what \nthou and the other women thought to...,91,96,900787,900912,[],None,NaN
329375,yale.39002088679528.clean,15208828,raiment,And what \nthou and the other women thought to...,97,104,900787,900912,[],None,NaN


In [111]:
# merged_df['gender'] = merged_df['gender'] != 'they/them/their'
merged_df = merged_df[merged_df['gender'] != 'they/them/']


In [112]:
 merged_df.to_csv('merged_df.csv')

In [113]:
merged_df

,book_id,mention_id,term,sentence,start_idx,end_idx,sentence_start_idx,sentence_end_idx,adjectives,character_id,gender
0,wu.89097998645.clean,6971477,sheath,I set her down \nas a New England woman the fi...,119,125,173434,173598,[],2947,he/him/his
1,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],282,she/her
2,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],281,he/him/his
3,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],2947,he/him/his
4,wu.89097998645.clean,6971480,coat,"While I was mourning over my cloak, she \ncame...",163,167,174093,174336,[],186,she/her
...,...,...,...,...,...,...,...,...,...,...,...
329372,yale.39002088679528.clean,15208824,clothing,But instantly it seemed \nto me that the voice...,122,130,896937,897183,[bare],None,NaN
329373,yale.39002088679528.clean,15208826,garments,"And they answered that the other women also, \...",152,160,899605,899838,[],None,NaN
329374,yale.39002088679528.clean,15208827,linen,And what \nthou and the other women thought to...,91,96,900787,900912,[],None,NaN
329375,yale.39002088679528.clean,15208828,raiment,And what \nthou and the other women thought to...,97,104,900787,900912,[],None,NaN


In [114]:
from sentence_transformers import SentenceTransformer
import numpy as np

df = merged_df

# df = df[:5000]

model = SentenceTransformer('all-MiniLM-L6-v2')

df['sentence_emb'] = df['sentence'].apply(lambda x: model.encode(x))
df['adj_emb'] = df['adjectives'].apply(lambda x: model.encode(' '.join(x)) if isinstance(x, list) else model.encode(str(x)))
df['fashion_emb'] = df['term'].apply(lambda x: model.encode(' '.join(x)) if isinstance(x, list) else model.encode(str(x)))


In [115]:
df['combined_emb'] = df.apply(
    lambda row: np.concatenate([row['sentence_emb'], row['adj_emb'], row['fashion_emb']]),
    axis=1
)


In [116]:
df = df.dropna()


X = np.vstack(df['combined_emb'])
y = df['gender']  # or whatever your target column is


In [117]:

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=500)
clf.fit(X, y)


LogisticRegression(max_iter=500)

find which embeddings were most helpful

In [118]:
weights = clf.coef_[0]  # for binary classification

sent_weights = weights[:384]
adj_weights = weights[384:768]
fashion_weights = weights[768:]


In [119]:
mean_importance = {
    'sentence': np.mean(np.abs(sent_weights)),
    'adjectives': np.mean(np.abs(adj_weights)),
    'fashion': np.mean(np.abs(fashion_weights))
}
print(mean_importance)


{'sentence': np.float64(0.17291790391799702), 'adjectives': np.float64(0.1746973718696708), 'fashion': np.float64(0.14422958790859453)}


In [120]:
key_vector = fashion_weights / np.linalg.norm(fashion_weights)


In [121]:
from sklearn.metrics.pairwise import cosine_similarity

# Suppose you have a list of unique fashion terms
# unique_fashion_terms = (set((df['term'], [])))

unique_fashion_terms = []

for idx,row in df.iterrows():
    if row['term'] not in unique_fashion_terms:
        unique_fashion_terms.append(row['term'])
    

# Encode them
fashion_term_embs = np.array([model.encode(term) for term in unique_fashion_terms])

# Compute similarity to key_vector
similarities = cosine_similarity(fashion_term_embs, key_vector.reshape(1, -1)).flatten()

# Get the top distinguishing terms
top_indices = np.argsort(similarities)[-10:][::-1]
for i in top_indices:
    print(unique_fashion_terms[i], similarities[i])


tam 0.17869670523696052
boa 0.16234027353219088
peruke 0.1532538496589415
bolero 0.15254933469037352
mac 0.14753161411413826
middy 0.14540552106937363
tams 0.14490967212740974
dags 0.1417860172837142
ruche 0.14091323153147645
kaftan 0.1407074772984597


In [122]:
#  y_pred = model.predict(X_test)
# y_pred_proba = model.predict_proba(X_test)
# 
# print("\n" + "=" * 50)
# print("CLASSIFICATION REPORT")
# print("=" * 50)
# print(classification_report(y_test, y_pred))
# 
# print("\n" + "=" * 50)
# print("CONFUSION MATRIX")
# print("=" * 50)
# cm = confusion_matrix(y_test, y_pred)
# print(cm)
# 
# # Plot confusion matrix
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=model.classes_,
#             yticklabels=model.classes_)
# plt.title('Confusion Matrix')
# plt.ylabel('True Label')
# plt.xlabel('Predicted Label')
# plt.tight_layout()
# plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
# plt.show()
# 
# return y_pred, y_pred_proba

In [123]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [124]:
clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)


In [125]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.5585809754016778

Classification Report:
                  precision    recall  f1-score   support

     he/him/his       0.56      0.99      0.72     23597
        she/her       0.40      0.01      0.01     13342
they/them/their       0.00      0.00      0.00      5259

       accuracy                           0.56     42198
      macro avg       0.32      0.33      0.24     42198
   weighted avg       0.44      0.56      0.40     42198


Confusion Matrix:
 [[23477   120     0]
 [13247    94     1]
 [ 5240    19     0]]


In [126]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print("Cross-validated accuracy: %.3f ± %.3f" % (scores.mean(), scores.std()))


Cross-validated accuracy: 0.558 ± 0.000


In [127]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


In [128]:
clf = LogisticRegression(max_iter=500)
clf.fit(X_resampled, y_resampled)


LogisticRegression(max_iter=500)

In [129]:
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.3559647376652922
                 precision    recall  f1-score   support

     he/him/his       0.59      0.36      0.45     23597
        she/her       0.34      0.33      0.34     13342
they/them/their       0.14      0.40      0.21      5259

       accuracy                           0.36     42198
      macro avg       0.36      0.36      0.33     42198
   weighted avg       0.45      0.36      0.38     42198

